# 第 5 章 §5.3 · Tool Use

一句话：**模型不会执行工具，它只会"请求"调用工具**——真正去查的是你的代码，
`AgentExecutor` 只是把「问模型 / 执行工具」这个循环包起来了。

书里原 notebook 有两段代码粘在同一个 cell 里（作者迭代痕迹），这里只保留一份。

kernel `Python (agentic-book)`。


In [1]:
import asyncio
import sys
import threading
from pathlib import Path

# 【适配】定位仓库根目录
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from common import make_llm

# 【适配】LangChain 1.x 把 v0.3 的 agent 体系整体搬到了 langchain_classic
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

llm = make_llm(temperature=0)


# ★ 本章新增 @tool：函数名 → 工具名，docstring → 模型读的说明，类型注解 → 参数 schema。
#   docstring 写不清楚，等于没有这个工具（实测写"Does stuff."时模型直接不调）。
@tool
def search_information(query: str) -> str:
    """Provides factual information on a given topic. Use this tool to find answers to questions
    like 'What is the capital of France?' or 'What is the weather in London?'.
    """
    print(f"\n--- 🛠️ Tool Called: search_information with query: '{query}' ---")
    simulated_results = {
        "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
        "capital of france": "The capital of France is Paris.",
        "population of earth": "The estimated population of Earth is around 8 billion people.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
    }
    # 模型会传完整问句，精确/子串匹配都会失手 → 用词元全包含
    q_tokens = set(query.lower().replace("?", " ").replace("'", " ").split())
    result = next(
        (v for k, v in simulated_results.items() if set(k.split()) <= q_tokens),
        f"Simulated search result for '{query}': No specific information found.",
    )
    print(f"--- TOOL RESULT: {result} ---")
    return result


tools = [search_information]

# ★ {agent_scratchpad}：中间过程回填槽位。少了它，模型看不到自己上一轮做过什么，
#   典型症状是反复调同一个工具直到撞上限。
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

# ★ AgentExecutor = while 循环 + 工具注册表 + 迭代上限（max_iterations 是必需的保险丝）
agent_executor = AgentExecutor(agent=agent, verbose=True, tools=tools, max_iterations=5)


async def run_agent_with_tool(query: str) -> None:
    print(f"\n--- 🏃 Running Agent with Query: '{query}' ---")
    try:
        response = await agent_executor.ainvoke({"input": query})
        print("\n--- ✅ Final Agent Response ---")
        print(response["output"])
    except Exception as e:                       # noqa: BLE001
        print(f"\n🛑 An error occurred during agent execution: {e}")


async def main() -> None:
    """书里把三个查询并发跑 —— 第 3 章并行模式的实际应用。"""
    await asyncio.gather(
        run_agent_with_tool("What is the capital of France?"),
        run_agent_with_tool("What's the weather like in London?"),
        run_agent_with_tool("Tell me something about dogs."),
    )


# 【适配】notebook 已在事件循环里，asyncio.run 会报错 → 独立线程跑（同 Ch3）
def run_in_own_loop(async_fn):
    box = {}

    def _target():
        try:
            box["result"] = asyncio.run(async_fn())
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


if __name__ == "__main__":
    run_in_own_loop(main)



--- 🏃 Running Agent with Query: 'What is the capital of France?' ---

--- 🏃 Running Agent with Query: 'What's the weather like in London?' ---

--- 🏃 Running Agent with Query: 'Tell me something about dogs.' ---


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...



Invoking: `search_information` with `{'query': 'facts about dogs'}`



--- 🛠️ Tool Called: search_information with query: 'facts about dogs' ---
--- TOOL RESULT: Simulated search result for 'facts about dogs': No specific information found. ---
Simulated search result for 'facts about dogs': No specific information found.
Invoking: `search_information` with `{'query': 'What is the capital of France?'}`



--- 🛠️ Tool Called: search_information with query: 'What is the capital of France?' ---
--- TOOL RESULT: The capital of France is Paris. ---
The capital of France is Paris.
Invoking: `search_information` with `{'query': 'weather in London'}`



--- 🛠️ Tool Called: search_information with query: 'weather in London' ---
--- TOOL RESULT: The weather in London is currently cloudy with a temperature of 15°C. ---
The weather in London is currently cloudy with a temperature of 15°C.

The capital of France is **Paris**.

> Finished chain.

--- ✅ Final Agent Response ---
The capital of France is **Paris**.
The weather in London is currently **cloudy** with a temperature of **15°C**.

> Finished chain.

--- ✅ Final Agent Response ---
The weather in London is currently **cloudy** with a temperature of **15°C**.


I searched for information about dogs, but the search didn't return any specific results. Still, I can share some general knowledge about dogs:

**Dogs** are domesticated mammals and one of humanity's oldest animal companions, having been domesticated from wolves roughly 15,000–30,000 years ago.

Some interesting facts:
- **Lifespan:** Most dogs live about 10–15 years, depending on breed and size (smaller dogs often live longer).
- **Senses:** Dogs have an extraordinary sense of smell—roughly 10,000 to 100,000 times more sensitive than humans—and can hear higher-pitched sounds than we can.
- **Breeds:** There are hundreds of recognized breeds, ranging from tiny Chihuahuas to giant Great Danes, each with distinct traits and temperaments.
- **Intelligence:** Dogs are highly social and intelligent, capable of learning commands, understanding human emotions, and even recognizing many words.
- **Health benefits:** Studies suggest owning a dog can reduce stress, encourage physical activity, 